# 📈 IA Trend Analyzer — Backend
### Corre esta única celda para levantar el servidor Flask completo
Asegúrate de tener tu `ANTHROPIC_API_KEY` en el archivo `.env` antes de correr.

In [ ]:
# ============================================================
# IA TREND ANALYZER — Backend completo en una sola celda
# Corre esta celda y el servidor Flask quedará activo
# ============================================================

import os, json, io, re, datetime, traceback
import pandas as pd
from flask import Flask, request, jsonify
from flask_cors import CORS
from dotenv import load_dotenv
import anthropic
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

load_dotenv()

# ── CONFIG ──────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = os.getenv('ANTHROPIC_MODEL', 'claude-opus-4-8')
GOOGLE_CREDS_FILE = os.getenv('GOOGLE_CREDENTIALS_FILE', 'credentials.json')
SCOPES = [
    'https://www.googleapis.com/auth/drive.file',
    'https://www.googleapis.com/auth/documents',
]

client_ai = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
app = Flask(__name__)
CORS(app, origins=["*"], supports_credentials=False)

current_df      = None
dataset_summary = ''

# ── GOOGLE AUTH ─────────────────────────────────────────────
def get_google_credentials():
    creds = None
    token_path = 'token.json'
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(GOOGLE_CREDS_FILE):
                raise FileNotFoundError(f'No se encontró {GOOGLE_CREDS_FILE}. Descárgalo desde Google Cloud Console.')
            flow = InstalledAppFlow.from_client_secrets_file(GOOGLE_CREDS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as f:
            f.write(creds.to_json())
    return creds

# ── HELPERS ──────────────────────────────────────────────────
def df_to_summary(df):
    buf = io.StringIO()
    buf.write(f'Columnas: {list(df.columns)}\n')
    buf.write(f'Filas: {len(df)}\n\n')
    buf.write('Estadísticas:\n')
    buf.write(df.describe(include='all').to_string())
    buf.write('\n\nPrimeras 5 filas:\n')
    buf.write(df.head(5).to_string())
    return buf.getvalue()

def ask_claude(system_prompt, user_message, history):
    messages = [{'role': h['role'], 'content': h['content']} for h in history]
    messages.append({'role': 'user', 'content': user_message})
    response = client_ai.messages.create(
        model=ANTHROPIC_MODEL, max_tokens=2048,
        system=system_prompt, messages=messages
    )
    return response.content[0].text

def extract_chart_data(df, analysis):
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    prompt = f"""
Análisis: {analysis}
Columnas numéricas: {numeric_cols}
Genera JSON PURO (sin markdown) con datos para 2 gráficas:
{{"chart1":{{"type":"line","label":"nombre","labels":[...],"data":[...]}},
  "chart2":{{"type":"bar","label":"nombre","labels":[...],"data":[...]}}}}
Solo JSON."""
    try:
        raw = client_ai.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=600,
            messages=[{'role': 'user', 'content': prompt}]
        ).content[0].text.strip()
        raw = re.sub(r'```(?:json)?', '', raw).strip().rstrip('`').strip()
        return json.loads(raw)
    except:
        return {
            'chart1': {'type': 'line',  'label': 'Tendencia',    'labels': ['Q1','Q2','Q3','Q4'], 'data': [40,65,55,80]},
            'chart2': {'type': 'bar',   'label': 'Distribución', 'labels': ['A','B','C','D'],      'data': [25,40,30,50]},
        }

def create_google_doc(title, content, chart_data):
    creds = get_google_credentials()
    docs  = build('docs', 'v1', credentials=creds)
    doc   = docs.documents().create(body={'title': title}).execute()
    doc_id = doc['documentId']
    now = datetime.datetime.now().strftime('%d/%m/%Y %H:%M')
    body_text = f"{title}\nGenerado por IA Trend Analyzer · {now}\n\n{'='*60}\n\n{content}\n\n{'='*60}\nGráfica 1 — {chart_data.get('chart1',{}).get('label','')}\nDatos: {chart_data.get('chart1',{}).get('data',[])}\n\nGráfica 2 — {chart_data.get('chart2',{}).get('label','')}\nDatos: {chart_data.get('chart2',{}).get('data',[])}\n"
    docs.documents().batchUpdate(
        documentId=doc_id,
        body={'requests': [{'insertText': {'location': {'index': 1}, 'text': body_text}}]}
    ).execute()
    return f'https://docs.google.com/document/d/{doc_id}/edit'

# ── ENDPOINTS ────────────────────────────────────────────────
@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': ANTHROPIC_MODEL})

@app.route('/upload', methods=['POST'])
def upload():
    global current_df, dataset_summary
    if 'file' not in request.files:
        return jsonify({'success': False, 'error': 'No se envió archivo'}), 400
    f    = request.files['file']
    name = f.filename.lower()
    try:
        if   name.endswith('.csv'):          df = pd.read_csv(f)
        elif name.endswith(('.xlsx','.xls')): df = pd.read_excel(f)
        elif name.endswith('.json'):          df = pd.read_json(f)
        else: return jsonify({'success': False, 'error': 'Formato no soportado'}), 400
        current_df = df
        dataset_summary = df_to_summary(df)
        return jsonify({'success': True, 'rows': len(df), 'columns': df.columns.tolist(), 'preview': df.head(3).to_dict(orient='records')})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/chat', methods=['POST'])
def chat():
    global current_df, dataset_summary
    data    = request.get_json()
    message = data.get('message', '')
    history = data.get('history', [])
    search_keywords = ['busca','encuentra','obtén','dame datos','buscar datos']
    is_search = any(kw in message.lower() for kw in search_keywords)
    if is_search and current_df is None:
        system = 'Eres un analista de datos. Genera un dataset representativo en JSON. Responde SOLO con JSON: {"response":"...","dataset":{"col1":[...],"col2":[...],...},"rows":20,"columns":[...]}'
        try:
            raw = client_ai.messages.create(
                model=ANTHROPIC_MODEL, max_tokens=1500,
                system=system, messages=[{'role':'user','content':message}]
            ).content[0].text.strip()
            raw = re.sub(r'```(?:json)?','',raw).strip().rstrip('`').strip()
            parsed = json.loads(raw)
            current_df = pd.DataFrame(parsed.get('dataset', {}))
            dataset_summary = df_to_summary(current_df)
            return jsonify({'success': True, 'response': parsed.get('response','Dataset generado.'), 'is_search_request': True, 'dataset_info': {'rows': len(current_df), 'columns': current_df.columns.tolist()}})
        except Exception as e:
            return jsonify({'success': False, 'error': str(e)}), 500
    dataset_context = 'DATASET ACTUAL:\n' + dataset_summary if dataset_summary else 'No hay dataset. Pide al usuario que suba uno.'
    system_prompt = f'Eres IA Trend Analyzer, analista de datos experto. Habla en el idioma del usuario. Usa negrillas para métricas importantes.\n\n' + dataset_context
    try:
        response = ask_claude(system_prompt, message, history[:-1])
        charts   = None
        if current_df is not None and any(kw in message.lower() for kw in ['tendencia','ventas','mejor','peor','promedio','total','análisis']):
            charts = extract_chart_data(current_df, response)
        return jsonify({'success': True, 'response': response, 'charts': charts})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/generate_report', methods=['POST'])
def generate_report():
    global current_df, dataset_summary
    data    = request.get_json()
    message = data.get('message', 'Genera un reporte completo')
    history = data.get('history', [])
    if current_df is None:
        return jsonify({'success': False, 'error': 'No hay dataset cargado'}), 400
    system_prompt = f"""Eres IA Trend Analyzer. Genera un reporte ejecutivo profesional.\nDATASET:\n{dataset_summary}\nIncluye: 1.RESUMEN EJECUTIVO 2.MÉTRICAS CLAVE 3.MEJOR RENDIMIENTO 4.TENDENCIAS 5.PROYECCIONES 6.RECOMENDACIONES\nAl final agrega: STATS_JSON: {{"metrica1":"valor1","metrica2":"valor2","metrica3":"valor3"}}"""
    try:
        response = ask_claude(system_prompt, message, history[:-1])
        charts   = extract_chart_data(current_df, response)
        stats    = {}
        m = re.search(r'STATS_JSON:\s*(\{.*?\})', response, re.DOTALL)
        if m:
            try: stats = json.loads(m.group(1)); response = response.replace(m.group(0),'').strip()
            except: pass
        title     = f"IA Trend Analyzer — Reporte {datetime.datetime.now().strftime('%d-%m-%Y %H:%M')}"
        drive_url = create_google_doc(title, response, charts)
        return jsonify({'success': True, 'response': response, 'charts': charts, 'stats': stats, 'drive_url': drive_url})
    except FileNotFoundError as e:
        return jsonify({'success': False, 'error': f'Google Drive no configurado: {e}'}), 500
    except Exception as e:
        traceback.print_exc()
        return jsonify({'success': False, 'error': str(e)}), 500

# ── ARRANCAR SERVIDOR ─────────────────────────────────────────
print('🚀 IA Trend Analyzer corriendo en http://localhost:5000')
print(f'   Modelo: {ANTHROPIC_MODEL}')
app.run(host='0.0.0.0', port=5001, debug=False, use_reloader=False)


🚀 IA Trend Analyzer corriendo en http://localhost:5000
   Modelo: claude-opus-4-8
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://192.168.1.12:5001
Press CTRL+C to quit
127.0.0.1 - - [06/Jun/2026 15:40:18] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:25] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:27] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:32] "OPTIONS /chat HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:35] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:37] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:42] "POST /upload HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:43] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:48] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:48] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:40:52] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:41:00] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [06/Jun/2026 15:41:08] "GET /health HTTP/1.1" 200 -
127.0.